# Chattbottar - Kapitel 10 (Lokal LLM)

I detta kodexempel ska vi få en grundläggande förståelse för hur chattbottar fungerar med **lokala modeller** (ingen API-nyckel behövs!). Let's go!

In [1]:
# Installera nödvändiga bibliotek för lokala modeller
# pip install transformers torch sentence-transformers
# pip install pypdf

In [2]:
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from sentence_transformers import SentenceTransformer
from pypdf import PdfReader
import torch
import warnings
warnings.filterwarnings('ignore')

C:\Users\daedi\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Använda en chattbot med lokal modell
Vi kommer nu demonstrera hur vi kan ha en egen chattbot genom att använda en **lokal modell** som körs på din egen dator. Detta kräver **ingen API-nyckel** och är helt gratis!

**Fördelar med lokala modeller:**
- Ingen API-nyckel behövs
- Helt gratis (ingen kostnad per anrop)
- Bättre integritet (data stannar på din dator)
- Fungerar offline
- Snabbare (ingen nätverksfördröjning)

**Nackdelar:**
- Kräver mer minne (modeller kan vara stora, 2-20 GB+)
- Kräver mer processorkraft (GPU rekommenderas för större modeller)
- Sämre prestanda än större molnmodeller

I detta exempel använder vi `gpt2` som är en lättviktig modell (~500 MB) som fungerar bättre med svenska text än DialoGPT (som är tränad på engelska konversationer).

In [ ]:
# Ladda ner och initiera lokal modell (görs bara en gång)
# Detta kan ta några minuter första gången eftersom modellen laddas ner
print("Laddar ner lokal modell... Detta kan ta några minuter första gången.")

# Använd GPT-2 istället för DialoGPT (fungerar bättre med svenska)
# GPT-2 är mer generell och hanterar svenska text bättre än DialoGPT som är tränad på engelska konversationer
model_name = "gpt2"  # GPT-2 fungerar bättre med svenska än DialoGPT
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Sätt pad_token (GPT-2 behöver detta)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Om du har GPU, använd den för snabbare inferens
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
print(f"Modell laddad på: {device}")

# Funktion för att generera svar med lokal modell
# GPT-2 fungerar bra både med och utan konversationshistorik
def generate_content_local(prompt, chat_history_ids=None, max_length=100, temperature=0.7, max_input_length=512):
    """
    Genererar svar med lokal GPT-2-modell.
    
    Args:
        prompt: Texten att generera svar från
        chat_history_ids: Tidigare konversationshistorik (torch tensor), None för första meddelandet
        max_length: Maximal längd på genererat svar (i tokens)
        temperature: Kreativitet (högre = mer kreativt, lägre = mer deterministiskt)
        max_input_length: Maximal längd på input (för att undvika tokeniseringsfel)
    
    Returns:
        response: Genererat svar (str)
        new_history_ids: Uppdaterad konversationshistorik för nästa anrop (torch tensor)
    """
    # Sätt pad_token om den inte finns (DialoGPT behöver detta)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    # Trunkera prompt om den är för lång (för att undvika tokeniseringsfel)
    # Tokenisera för att kontrollera längd
    prompt_tokens = tokenizer.encode(prompt, add_special_tokens=False)
    if len(prompt_tokens) > max_input_length:
        # Trunkera från början (behåll slutet som är viktigast)
        prompt_tokens = prompt_tokens[-max_input_length:]
        prompt = tokenizer.decode(prompt_tokens, skip_special_tokens=True)
    
    # Tokenisera input med EOS token (använd truncation för säkerhet)
    new_user_input_ids = tokenizer.encode(
        prompt + tokenizer.eos_token, 
        return_tensors="pt",
        truncation=True,
        max_length=max_input_length
    ).to(device)
    
    # Kombinera med historik om den finns
    if chat_history_ids is not None:
        # Kontrollera total längd
        total_length = chat_history_ids.shape[-1] + new_user_input_ids.shape[-1]
        max_total_length = model.config.max_position_embeddings if hasattr(model.config, 'max_position_embeddings') else 1024
        
        if total_length > max_total_length - max_length:
            # Trunkera historik om den är för lång
            keep_length = max_total_length - max_length - new_user_input_ids.shape[-1] - 50  # Lämna lite marginal
            if keep_length > 0:
                chat_history_ids = chat_history_ids[:, -keep_length:]
            else:
                # Om input är för lång, använd bara input
                chat_history_ids = None
        
        if chat_history_ids is not None:
            bot_input_ids = torch.cat([chat_history_ids, new_user_input_ids], dim=-1)
        else:
            bot_input_ids = new_user_input_ids
    else:
        bot_input_ids = new_user_input_ids
    
    # Beräkna total max_length (input + nytt svar)
    input_length = bot_input_ids.shape[-1]
    max_total_length = model.config.max_position_embeddings if hasattr(model.config, 'max_position_embeddings') else 1024
    total_max_length = min(input_length + max_length, max_total_length)
    
    # Generera svar
    with torch.no_grad():
        try:
            chat_history_ids = model.generate(
                bot_input_ids,
                max_length=total_max_length,
                temperature=temperature,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
                no_repeat_ngram_size=3,  # Undvik repetering
                num_return_sequences=1
            )
        except Exception as e:
            print(f"Fel vid generering: {e}")
            # Fallback: försök med kortare input
            if bot_input_ids.shape[-1] > 100:
                bot_input_ids = bot_input_ids[:, -100:]
                input_length = bot_input_ids.shape[-1]
                total_max_length = min(input_length + max_length, max_total_length)
                chat_history_ids = model.generate(
                    bot_input_ids,
                    max_length=total_max_length,
                    temperature=temperature,
                    do_sample=True,
                    pad_token_id=tokenizer.eos_token_id,
                    num_return_sequences=1
                )
            else:
                raise
    
    # Extrahera bara det nya svaret (ta bort input och historik)
    response_ids = chat_history_ids[:, input_length:]
    response = tokenizer.decode(response_ids[0], skip_special_tokens=True)
    
    return response, chat_history_ids

# Testa modellen
prompt = "Hej! Kan du förklara vad AI är?"
response, _ = generate_content_local(prompt, max_length=150)
print(response)

Laddar ner lokal modell... Detta kan ta några minuter första gången.
Modell laddad på: cpu
How do you feel about the way things are going in your house?

I was pretty happy with the first time I was in there. I was trying to figure out a way to make it work. I had been looking for a way that would make my house look cool and clean and had recently been looking at the internet for a better way. And that's what I found.

It was so easy to get here. I just came down from the sky and was about to put my big box of ice cream in the freezer. I picked it up and was like, "Oh my gosh, this is a recipe!" When I opened it I was like wow, this makes your house look really cool. It also smells really


# Skapa en mycket enkel applikation
Vi kan också skapa en enkel applikation. Med lite kreativitet så inser man att det finns många möjligheter att bygga/utveckla sådant man är intresserad av.

In [4]:
print("*** Lokal chattbot ***")
print("Type <q> to exit chat.")

# Initiera konversationshistorik (DialoGPT behöver detta för att fungera bra)
chat_history_ids = None

while True:
    prompt = input("User: ").strip()
    
    # Avsluta om användaren skriver 'q' eller tom sträng
    if not prompt or prompt.lower() == "q":
        break
    
    try:
        # Använd lokal modell med konversationshistorik (ingen API-nyckel behövs!)
        response, chat_history_ids = generate_content_local(
            prompt, 
            chat_history_ids=chat_history_ids, 
            max_length=100  # Längd på genererat svar
        )
        print("Bot:", response)
    except Exception as e:
        print(f"Fel: {e}")
        print("Försöker igen...")
        # Återställ historik vid fel
        chat_history_ids = None

*** Lokal chattbot ***
Type <q> to exit chat.


# Retrieval Augmented Generation (RAG)

## Läsa in PDF-fil

Vi börjar med att läsa in en PDF-fil som chattbotten kommer ge svar utifrån.

In [5]:
reader = PdfReader("chattbot.pdf")

text = ""
for page in reader.pages:
    text += page.extract_text()

In [6]:
print(len(text))

23416


In [7]:
print(text[17:550])

Kapitel 1
Chattbot
1.1 Chattbottar
I detta kapitel kommer vi lära oss mer om hur chattbottar såsom ChatGPT fungerar och
hur man ställer effektiva frågor till dem, vilket kallas prompt engineering . Vi kommer
därefter lära oss hur vi bygger en lokal chattbot som vi kan använda på vår egna dator.
Kapitlet avslutas med att gå igenom RAG vilket låter oss anpassa chattbottens svar
utifrån en given kontext, exempelvis utifrån egna dokument.
1.1.1 ChatGPT och prompt engineering
Chattbottar, som till exempel ChatGPT, används av många m


## Chunking

### Fixed length-chunking

In [8]:
chunks = []
n = 1000
overlap = 200
for i in range(0, len(text), n - overlap):
    chunks.append(text[i:i + n])

print(f"Antal chunks: {len(chunks)}.")

Antal chunks: 30.


In [9]:
print(chunks[0])

Chattbot med RAG2Kapitel 1
Chattbot
1.1 Chattbottar
I detta kapitel kommer vi lära oss mer om hur chattbottar såsom ChatGPT fungerar och
hur man ställer effektiva frågor till dem, vilket kallas prompt engineering . Vi kommer
därefter lära oss hur vi bygger en lokal chattbot som vi kan använda på vår egna dator.
Kapitlet avslutas med att gå igenom RAG vilket låter oss anpassa chattbottens svar
utifrån en given kontext, exempelvis utifrån egna dokument.
1.1.1 ChatGPT och prompt engineering
Chattbottar, som till exempel ChatGPT, används av många människor för en rad olika
syften. Alltifrån att skapa kreativt innehåll, få förslag på förbättringar av skriven text
eller programmeringskod till att lösa mer komplexa problem.
I korthet har chattbottar tränats på enorma datamängder och har från detta lärt sig vad
som är rimliga sekvenser av ord. Exempelvis vet vi att om någon säger “Hej, hur mår
___” så brukar det sista ordet vara “du” för att meningen ska bli “Hej, hur mår du” .
När vi ställer 

In [10]:
print(text[26:584])


Chattbot
1.1 Chattbottar
I detta kapitel kommer vi lära oss mer om hur chattbottar såsom ChatGPT fungerar och
hur man ställer effektiva frågor till dem, vilket kallas prompt engineering . Vi kommer
därefter lära oss hur vi bygger en lokal chattbot som vi kan använda på vår egna dator.
Kapitlet avslutas med att gå igenom RAG vilket låter oss anpassa chattbottens svar
utifrån en given kontext, exempelvis utifrån egna dokument.
1.1.1 ChatGPT och prompt engineering
Chattbottar, som till exempel ChatGPT, används av många människor för en rad olika
syften. 


## Embeddings
Se t.ex. s.321 i kursboken "Lär dig AI från grunden - Tillämpad maskininlärning med Python" för vad Embeddings innebär. I korthet, ord representeras med vektorer/siffror.

**Med lokal modell:** Vi använder `sentence-transformers` som är ett bibliotek för att skapa embeddings lokalt utan API-nyckel.

In [11]:
# Ladda lokal embedding-modell (görs bara en gång)
# Detta kan ta några minuter första gången
print("Laddar ner lokal embedding-modell...")

embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"  # Snabb och lättviktig (~80 MB)
embedding_model = SentenceTransformer(embedding_model_name)
print("Embedding-modell laddad!")

def create_embeddings_local(texts):
    """
    Skapar embeddings för texter med lokal modell.
    
    Args:
        texts: En text eller lista med texter
    
    Returns:
        Embeddings som numpy-array
    """
    # Om det är en enskild text, gör om till lista
    if isinstance(texts, str):
        texts = [texts]
    
    # Skapa embeddings
    embeddings = embedding_model.encode(texts, convert_to_numpy=True)
    
    return embeddings

Laddar ner lokal embedding-modell...
Embedding-modell laddad!


In [12]:
# Skapa embeddings för alla chunks (detta kan ta några sekunder)
print("Skapar embeddings för alla chunks...")
chunk_embeddings = create_embeddings_local(chunks)
print(f"Antal embeddings: {len(chunk_embeddings)}")
print(f"Dimension på varje embedding: {chunk_embeddings[0].shape}")

Skapar embeddings för alla chunks...
Antal embeddings: 30
Dimension på varje embedding: (384,)


In [13]:
# Visa första 10 värdena i första embedding-vektorn
chunk_embeddings[0][0:10]

array([-0.0117534 ,  0.01186229,  0.02151772, -0.05307685,  0.00766778,
       -0.05013887,  0.06382428,  0.11596681,  0.03580072,  0.03167636],
      dtype=float32)

## Semantisk sökning

In [14]:
def cosine_similarity(vec1, vec2):
    """
    Beräknar cosinuslikhet mellan två vektorer.
    
    Args:
        vec1, vec2: Numpy arrays eller listor med samma längd
    
    Returns:
        Cosinuslikhet (värde mellan -1 och 1, där 1 = identiska)
    """
    return (np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2)))

In [15]:
def semantic_search(query, chunks, chunk_embeddings, k=5):
    """
    Söker efter de k mest relevanta chunks baserat på semantisk likhet.
    
    Args:
        query: Sökfrågan
        chunks: Lista med text-chunks
        chunk_embeddings: Embeddings för chunks (numpy array)
        k: Antal toppresultat att returnera
    
    Returns:
        Lista med de k mest relevanta chunks
    """
    # Skapa embedding för frågan med lokal modell
    query_embedding = create_embeddings_local(query)[0]  # [0] eftersom vi får en lista tillbaka
    
    # Beräkna likhet mellan frågan och alla chunks
    similarity_scores = []
    
    for i, chunk_embedding in enumerate(chunk_embeddings):
        similarity_score = cosine_similarity(query_embedding, chunk_embedding)
        similarity_scores.append((i, similarity_score))

    # Sortera efter likhet (högst först)
    similarity_scores.sort(key=lambda x: x[1], reverse=True)
    
    # Hämta de k bästa indices
    top_indices = [index for index, _ in similarity_scores[:k]]
    
    # Returnera motsvarande chunks
    return [chunks[index] for index in top_indices]

In [ ]:
fråga = "Vad kan RAG användas till?"
svar = semantic_search(fråga, chunks=chunks, chunk_embeddings=chunk_embeddings, k=1)
print(svar[0])

nvända RAG-
tekniken för att begränsa en språkmodell till att enbart generera svar utifrån en given
kontext.
Vi har också sett exempel på hur vi kan skapa en vector store för att hantera och lagra
våra embeddings.
20


## Generera bra svar med RAG

In [17]:
system_prompt = """Jag kommer ställa dig en fråga, och jag vill att du svarar
baserat bara på kontexten jag skickar med, och ingen annan information.
Om det inte finns nog med information i kontexten för att svara på frågan,
säg "Det vet jag inte". Försök inte att gissa.
Formulera dig enkelt och dela upp svaret i fina stycken. """

In [18]:
def generate_user_prompt(query):
    """
    Skapar en user prompt med kontext från semantisk sökning.
    
    Args:
        query: Användarens fråga
    
    Returns:
        En formaterad prompt med frågan och relevant kontext
    """
    # Hitta relevanta chunks
    context = "\n".join(semantic_search(query, chunks, chunk_embeddings))
    
    # Kombinera frågan med kontexten
    user_prompt = f"Frågan är {query}. Här är kontexten: {context}."
    
    return user_prompt

In [19]:
def generate_response(system_prompt, user_message, max_length=300):
    """
    Genererar svar med RAG med lokal modell.
    
    Args:
        system_prompt: Systeminstruktioner för modellen
        user_message: Användarens fråga
        max_length: Maximal längd på genererat svar
    
    Returns:
        Ett Response-objekt med .text attribut
    """
    # Kombinera system prompt och user prompt
    full_prompt = f"{system_prompt}\n\n{generate_user_prompt(user_message)}"
    
    # Generera svar med lokal modell (ingen historik för RAG, varje fråga är oberoende)
    # Använd kortare max_input_length för RAG eftersom prompts kan vara långa
    response_text, _ = generate_content_local(
        full_prompt, 
        chat_history_ids=None, 
        max_length=max_length, 
        temperature=0.7,
        max_input_length=400  # Begränsa input-längd för RAG
    )
    
    # Skapa ett enkelt Response-objekt för kompatibilitet
    class Response:
        def __init__(self, text):
            self.text = text
    
    return Response(response_text)

In [20]:
print(generate_response(system_prompt, "Vad är RAG?").text)

Token indices sequence length is longer than the specified maximum sequence length for this model (1932 > 1024). Running this sequence through the model will result in indexing errors



18Väser av ava år det det dock det vändri mångsta och havnade.
19Där det och det det bøg med därs var i det på det det en månskligt.
20Tår det som det det det ock om med det növäng, det det efter och och
det det det.
22Därr det det tær det med det det hår kommt. Det det det vårde det det månändre. Det med det anvændt det det
set det. Närsker det det, det var och männade. Det var det det att det det kommen. Det tår
väst det det med. Det
22Tær med det töndr det märtå det pöllen.
23Därd det det var det. Det i det det närsklig. Det vekt det det anmår. Det väst tært det det pällen, det tåt det var
re det det ald det detvar det detter det dett dett til det detkomme dett. det det svar det
det tæf det det set


In [35]:
fråga = "Vad är meningen med livet?"
svar = generate_response(system_prompt, fråga).text
print(svar)

Token indices sequence length is longer than the specified maximum sequence length for this model (1920 > 1024). Running this sequence through the model will result in indexing errors



13 själle.
jag komme det äre vål av varna och det
brella kommed att kanke för
jäg äber att kanak jag konte
äre det det det kanke att
"Norse: The Red List"
14 ufendet.
varna ufende skaggard.
Som håd och ärik det bier hätter med
jak på det det att kanek .
jorgt att kanki enkelet att kanker
"Dietary Stem"
15 håjäst.
diet i kompen i gåningar om varna såd som
et det att det kanka .
på det att med kanter med kanker.
Til kan kommen händ på parjæta
et kan kär än den
tär det kankom på näru
"No"
16 hånskåsen.
tå det kanter hån varna på kanker och
"I Love a Girl"
17 och som det att mandar kommand.
För det kan komplæk att kank och av
tak som det kanker


In [22]:
fråga = "Vad är prompt engineering?"
svar = generate_response(system_prompt, fråga).text
print(svar)

 säga med . detta detta sam
de om den komppod för den sverdamäninger
der vi, med til detta med
de att. gå som ikvällas ikkom på månnisker
då nach en den kor för med och detta
så att detta ävänte ik vad, detta att deta som
tälker det ik älker som vi
komppart til deta er sven ik detta någa
för att dett.
Så vad er om detta ik er detta er deta ik ik komplätter
vad. detta . deta detta om deta vad om dettavände
vas ikan, dettakon med detta svergade
en deta . dett avs av detta tät kor
vaväntet var med dett
vatn att dets
sekvenrödta dettvakon
vätak en att det
vast er dettad
vårentet ik och för detta var
vasset om om detva ik
kommppart det


# Evaluering

In [23]:
validation_data = [
    {
        "question": "Vilka delar utgör en RAG-modell?",
        "ideal_answer": """En RAG-modell innehåller två delar: 
en retriever som söker efter relevanta stycken i en text, 
och en generator som genererar svar utifrån den givna kontexten."""
    }
]

print(validation_data[0]["question"])
print()
print(validation_data[0]["ideal_answer"])

Vilka delar utgör en RAG-modell?

En RAG-modell innehåller två delar: 
en retriever som söker efter relevanta stycken i en text, 
och en generator som genererar svar utifrån den givna kontexten.


In [24]:
evaluation_system_prompt = """Du är ett intelligent utvärderingssystem vars uppgift är att utvärdera en AI-assistents svar. 
Om svaret är väldigt nära det önskade svaret, sätt poängen 1. Om svaret är felaktigt eller inte bra nog, sätt poängen 0.
Om svaret är delvis i linje med det önskade svaret, sätt poängen 0.5. Motivera kort varför du sätter den poäng du gör.
"""

In [25]:
query = validation_data[0]["question"]

response = generate_response(system_prompt, query)

evaluation_prompt = f"""Fråga: {query}
AI-assistentens svar: {response.text}
Önskat svar: {validation_data[0]['ideal_answer']}"""

# Använd samma generate_response-funktion för evaluering
evaluation_response = generate_response(evaluation_system_prompt, evaluation_prompt)
print(evaluation_response.text)


Tõnnen om takker äveksen att upp, det ersten stälta ett, sällas och kan med det kan sälvas ossikt eller
hällat om stärtat om söldas.
Ett ergården om med det med kan sölvas takke kan utifragt.
Un karke om utifagt, det seldis om utfüllat.
Kontext kan utfårdon för och stölfet.
Värligt av av avan äkomme och att uppy.
Ñt hällar om med takken om med kan stälvat om
jällare om utfiagt om utpåt om.
En erpåntas om utflare om stölvat, det det kan ett ergen ställam.
10.
Hvor en om utfoagt ett om statt om utnärt, det nämtt av äjän takka ävällen om utprävär,
un kon texten növännen nivy,
et har äpp och ergen det st


In [26]:
# Try change to "ideal_answer": "Java" and notice that if the user defines the wrong "ideal answer", then it gets weird. 
# In a wider context, how do you define ideal answers to questions that have no exact answer and who does this? 

validation_data_2 = [
    {
        "question": "Vilket programmeringsspråk är kapitlet skrivet i?",
        "ideal_answer": "Python"
    }
]
print(validation_data_2[0]["question"])
print(validation_data_2[0]["ideal_answer"])

Vilket programmeringsspråk är kapitlet skrivet i?
Python


In [27]:
query = validation_data_2[0]["question"]

response = generate_response(system_prompt, query)
print(response.text)


(A komme skåkter det ska,
Ok på häll det av kom med såsten
and skan videt. Det
de skär att kan kommed.
är den södverlag med enkär sven,
Nikke er den såvär det det.
Nagtet er den vär den kvälånninger med
te enkelter om ska vid.
Därst välär nils gåt enkelva den kom
vällkter enkellter om sven.
The tigressen vällånner hält för
lehst av de skälöv och kom på den sven
alte med välvär manningar.
Och och oche vältet som det
de komkter sven och det det väla nälv
de för väld.
A kon texten nåläkkker äländet och
výlänt urske urskop.
Heväs enkulte mälte,
Konninger tussen om urskt


In [28]:
evaluation_prompt = f"""Fråga: {query}
AI-assistentens svar: {response.text}
Önskat svar: {validation_data_2[0]['ideal_answer']}"""

evaluation_response = generate_response(evaluation_system_prompt, evaluation_prompt)
print(evaluation_response.text)

0/1/4/4.pdf
Iktivsäktiv-tärksökte i den mänster äljorden sökta den dettänningen fört.
Skar sevällt med mödchen at som kraut.
We tikkert om sägällte
“Tikkert” ikta med männen. “A lot of stuff
”a lot of shit.
 “A LOT of shit …
‌“‌ „ “‐ „
—‐— “
‬‬ “ “ ̄
‒—
••
‖‖ ―‖
–―
‡‡
‫‫
‮‮
‽‒
″″
…‡ ‡‥
․․
‥‥ “


# Fördjupning
Den som är intresserad av att arbeta med chattbottar för t.ex. del 2 av kunskapskontrollen kan fördjupa sig inom LangChain, se t.ex. här: 
https://academy.langchain.com/collections/foundation

Vill man få en snabb överblick, se t.ex. "Quickstart LangChain Essentials - Python" här: https://academy.langchain.com/collections/quickstart

Vill man få en överblick över LangChain, LangGraph och LangSmith, se här: https://www.youtube.com/watch?v=vJOGC8QJZJQ

**För lokala modeller:** Du kan också utforska andra modeller på Hugging Face: https://huggingface.co/models